# Try BreadBowl Embed

Encode passages, inspect routing/value slots, and search with late interaction. Install this repository first (`python -m pip install .`). Until Hub weights are published, set `BREADBOWL_MODEL` to a local exported model directory before starting Jupyter. CPU is the verified default; this is a small example, not a benchmark.

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from breadbowl_embed import BreadBowl, Index

model = BreadBowl.from_pretrained(os.environ.get("BREADBOWL_MODEL", "dotproductx/breadbowl-embed-v1.2-preview"))

In [ ]:
passages = [
    "Octopuses have three hearts.",
    "The Pacific is Earth's largest ocean.",
    "Plants use sunlight during photosynthesis.",
]
ids = ["octopus", "ocean", "plants"]
documents = model.encode_documents(passages, batch_size=2)
queries = model.encode_queries(["How many hearts does an octopus have?"])
print("Routing:", documents.routing.shape)
print("Values:", documents.value.shape)

Routing slots are normalized. Document values retain their original magnitudes before the query-dependent attention read. Cache both tensors; a standard single-vector cosine index cannot reproduce this scorer.

In [ ]:
scores = model.score(queries, documents)
print("Routing:", scores.routing)
print("Value:", scores.value)
print("Combined:", scores.combined)
for hit in model.rerank("How many hearts does an octopus have?", documents, ids=ids):
    print(hit.id, hit.score, passages[hit.document_index])

In [ ]:
index = model.index(documents, ids=ids)
hits = index.search(queries, top_k=2, candidate_k=len(passages))
for hit in hits[0]:
    print(hit.to_dict())
with TemporaryDirectory() as temporary:
    directory = Path(temporary) / "index"
    index.save(directory)
    restored = Index.load(directory)
    assert restored.search(queries, top_k=2, candidate_k=len(passages)) == hits

The index scans all document routing slots and reranks a shortlist with routing and values. Here every document is a candidate. Use this for small experiments; it does not implement the paper's ANN/per-slot shortlist pipeline. Passage text is not stored in the index. Inputs exceeding the configured token limits are truncated.